In [252]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
import matplotlib.pyplot as plt
import seaborn as sns
import re
import string


In [253]:
import pandas as pd  
df = pd.read_csv("spam.csv", encoding='latin-1')



In [254]:
df.head()

,v1,v2,Unnamed: 2,Unnamed: 3,Unnamed: 4
0,ham,"Go until jurong point, crazy.. Available only ...",NaN,NaN,NaN
1,ham,Ok lar... Joking wif u oni...,NaN,NaN,NaN
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,NaN,NaN,NaN
3,ham,U dun say so early hor... U c already then say...,NaN,NaN,NaN
4,ham,"Nah I don't think he goes to usf, he lives aro...",NaN,NaN,NaN


In [255]:
df.drop(columns= ['Unnamed: 2' , 'Unnamed: 3','Unnamed: 4'], inplace= True)

In [256]:
df['clean_encode'] = df['v1'].map({ 'ham' : 0 , "spam": 1 })

In [257]:
df

,v1,v2,clean_encode
0,ham,"Go until jurong point, crazy.. Available only ...",0
1,ham,Ok lar... Joking wif u oni...,0
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,1
3,ham,U dun say so early hor... U c already then say...,0
4,ham,"Nah I don't think he goes to usf, he lives aro...",0
...,...,...,...
5567,spam,This is the 2nd time we have tried 2 contact u...,1
5568,ham,Will Ì_ b going to esplanade fr home?,0
5569,ham,"Pity, * was in mood for that. So...any other s...",0
5570,ham,The guy did some bitching but I acted like i'd...,0


In [258]:
df.isnull().sum()

v1              0
v2              0
clean_encode    0
dtype: int64

In [259]:
import re

def clean_text(text):
    text = text.lower()                        
    text = re.sub(r'[^a-z\s]', '', text)       
    text = re.sub(r'\s+', ' ', text).strip()    
    return text

In [260]:
df['clean_msg'] = df['v2'].apply(clean_text )

In [261]:
df['v1'].isnull().sum()

np.int64(0)

In [262]:
X_train, X_test, y_train, y_test = train_test_split(
    df['clean_msg'], df['clean_encode'], test_size=0.2, random_state=42, stratify=df['clean_encode']
)

In [263]:

vectorizer = TfidfVectorizer(stop_words='english', max_features=3000)
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

In [264]:
model = MultinomialNB()
model.fit(X_train_vec, y_train)

,alpha,1.0
,force_alpha,True
,fit_prior,True
,class_prior,None


In [265]:
y_pred = model.predict(X_test_vec)


In [266]:
print( accuracy_score(y_pred , y_test))

0.9713004484304932


In [267]:
cm = confusion_matrix(y_test, y_pred)
cm

array([[965,   1],
       [ 31, 118]])

In [268]:
def predict_message(msg):
    clean = clean_text(msg)
    vec = vectorizer.transform([clean])
    pred = model.predict(vec)[0]
    prob = model.predict_proba(vec)[0]
    = 'Spam' if pred == 1 else 'ham'   
    print(f"Message: {msg}")
    print(f"Prediction: {v1} (Confidence: {max(prob)*100:.2f}%)")



In [274]:
predict_message("Hey, are we still meeting for lunch tomorrow?")


Message: Hey, are we still meeting for lunch tomorrow?
Prediction: Ham (Confidence: 99.65%)


In [275]:
predict_message('URGENT! Your account has been suspended. Verify your details immediately by clicking this link to avoid permanent closure.') 

Message: URGENT! Your account has been suspended. Verify your details immediately by clicking this link to avoid permanent closure.
Prediction: Ham (Confidence: 74.11%)
